# Prediksi Kualitas Udara Jakarta
### Random Forest Classifier — Data ISPU DKI Jakarta 2024–2025

---
| Tahap | Isi |
|---|---|
| **Tahap 1** | Setup Library & Load Data |
| **Tahap 2** | Data Cleaning & Imputasi |
| **Tahap 3** | Exploratory Data Analysis (EDA) |
| **Tahap 4** | Feature Engineering |
| **Tahap 5** | Split Data & Pelatihan Model |
| **Tahap 6** | Evaluasi Model |
| **Tahap 7** | Validasi (Stratified K-Fold) |
| **Tahap 8** | Simulasi Prediksi Data Baru |

> **Fitur:** PM10, PM2.5, SO2, CO, O3, NO2, Stasiun (One-Hot Encoding)  
> **Target:** Kategori kualitas udara (BAIK, SEDANG, TIDAK SEHAT)

## Tahap 1 — Setup Library & Load Data

In [ ]:
import pandas as pd
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

DATA_DIR = Path(r".\dataset")

# Nama file dataset
TARGET_FILE = "Data_Indeks Standar Pencemar Udara (ISPU) di Provinsi DKI Jakarta - tabel.xls"
file_path   = DATA_DIR / TARGET_FILE

try:
    df_raw = pd.read_html(file_path)[0]
    df_raw.columns = df_raw.columns.str.lower().str.strip()
    print(f"Berhasil memuat: {TARGET_FILE}")
    print(f"Jumlah data    : {df_raw.shape[0]} baris, {df_raw.shape[1]} kolom")
except Exception as e:
    print(f"Gagal memproses file: {e}")

## Tahap 2 — Data Cleaning & Imputasi

In [ ]:
import numpy as np

print("=== TAHAP 2: DATA CLEANING ===")

# Standarisasi nama kolom ke format baku
def standarisasi_kolom(df):
    kolom_mapping = {
        'wilayah'                  : 'stasiun',
        'lokasi_spku'              : 'stasiun',
        'pm_sepuluh'               : 'pm10',
        'pm_duakomalima'           : 'pm25',
        'sulfur_dioksida'          : 'so2',
        'karbon_monoksida'         : 'co',
        'ozon'                     : 'o3',
        'nitrogen_dioksida'        : 'no2',
        'parameter_pencemar_kritis': 'critical',
        'categori'                 : 'kategori'
    }
    df_renamed = df.rename(columns=kolom_mapping)
    new_cols = {}
    for col in df_renamed.columns.unique():
        if isinstance(df_renamed[col], pd.DataFrame):
            new_cols[col] = df_renamed[col].bfill(axis=1).iloc[:, 0]
        else:
            new_cols[col] = df_renamed[col]
    return pd.DataFrame(new_cols)

# Pembersihan isi data: stasiun, polutan, dan target
def clean_isi_data(df):
    df = df.copy()

    # Standarisasi nama stasiun ke kode DKI1-DKI5
    def bersihkan_stasiun(nama):
        nama = str(nama).upper()
        if 'DKI1' in nama: return 'DKI1'
        elif 'DKI2' in nama: return 'DKI2'
        elif 'DKI3' in nama: return 'DKI3'
        elif 'DKI4' in nama: return 'DKI4'
        elif 'DKI5' in nama: return 'DKI5'
        else: return np.nan

    df['stasiun'] = df['stasiun'].apply(bersihkan_stasiun)
    df = df.dropna(subset=['stasiun'])

    # Konversi polutan ke numerik & imputasi median per stasiun
    kolom_polutan = ['pm10', 'pm25', 'so2', 'co', 'o3', 'no2']
    for col in kolom_polutan:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
            df[col] = df.groupby('stasiun')[col].transform(lambda x: x.fillna(x.median()))

    # Bersihkan kolom target
    df = df.dropna(subset=['kategori'])
    df['kategori'] = df['kategori'].astype(str).str.upper().str.strip()

    # Pertahankan hanya 4 kategori ISPU yang valid
    kategori_valid = ['BAIK', 'SEDANG', 'TIDAK SEHAT', 'SANGAT TIDAK SEHAT']
    df = df[df['kategori'].isin(kategori_valid)]

    # Hapus kolom yang tidak dipakai sebagai fitur (mencegah data leakage)
    kolom_buang = ['periode_data', 'tanggal', 'bulan', 'max', 'critical', 'source_file']
    df = df.drop(columns=[col for col in kolom_buang if col in df.columns])

    return df

df_clean = clean_isi_data(standarisasi_kolom(df_raw))

print("=" * 50)
print(f"Jumlah data bersih : {df_clean.shape[0]} baris")
print(f"Kolom              : {df_clean.columns.tolist()}")
print(f"Distribusi kategori:")
print(df_clean['kategori'].value_counts().to_string())

## Tahap 3 — Exploratory Data Analysis (EDA)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

print("=== TAHAP 3: EXPLORATORY DATA ANALYSIS ===")
print(f"Total data: {df_clean.shape[0]} baris\n")

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Exploratory Data Analysis — ISPU DKI Jakarta',
             fontsize=16, fontweight='bold', y=1.02)

# Grafik 1: Distribusi kategori kualitas udara
sns.countplot(data=df_clean, x='kategori', ax=axes[0], palette='magma',
              order=['BAIK', 'SEDANG', 'TIDAK SEHAT', 'SANGAT TIDAK SEHAT'])
axes[0].set_title(f'Distribusi Kategori Kualitas Udara\n(Total: {len(df_clean)} data)', pad=10)
axes[0].set_xlabel('Kategori'); axes[0].set_ylabel('Jumlah Data')
axes[0].tick_params(axis='x', rotation=15)

# Grafik 2: Heatmap korelasi antar polutan
corr = df_clean.select_dtypes(include=['float64', 'int64']).corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f", ax=axes[1], square=True)
axes[1].set_title('Korelasi Antar Polutan (Termasuk PM2.5)', pad=10)

plt.tight_layout()
nama_eda = 'eda_ispu.png'
plt.savefig(nama_eda, dpi=300, bbox_inches='tight')
plt.show()
print(f"Grafik EDA disimpan: '{nama_eda}'")

## Tahap 4 — Feature Engineering

In [ ]:
print("=== TAHAP 4: FEATURE ENGINEERING ===")

# Mapping kategori ke angka ordinal
map_kategori         = {'BAIK': 0, 'SEDANG': 1, 'TIDAK SEHAT': 2, 'SANGAT TIDAK SEHAT': 3}
map_kategori_terbalik = {v: k for k, v in map_kategori.items()}

def siapkan_fitur_target(df):
    df_copy = df.copy()

    # Encoding label target ke angka ordinal
    df_copy['kategori'] = df_copy['kategori'].map(map_kategori)

    # One-Hot Encoding stasiun (drop_first menghindari dummy variable trap)
    df_copy = pd.get_dummies(df_copy, columns=['stasiun'], drop_first=True)

    # Konversi boolean hasil get_dummies ke integer
    bool_cols = df_copy.select_dtypes(include=['bool']).columns
    df_copy[bool_cols] = df_copy[bool_cols].astype(int)

    X = df_copy.drop(columns=['kategori'])
    y = df_copy['kategori']
    return X, y

X, y = siapkan_fitur_target(df_clean)

print(f"Dimensi fitur  : {X.shape}")
print(f"Dimensi target : {y.shape}")
print(f"Kolom fitur    : {X.columns.tolist()}")
print("\n3 baris pertama fitur:")
display(X.head(3))

## Tahap 5 — Split Data & Pelatihan Model

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

print("=== TAHAP 5: SPLIT DATA & PELATIHAN MODEL ===")

# Hapus kelas yang terlalu sedikit untuk dibagi (< 2 sampel)
counts       = y.value_counts()
kelas_valid  = counts[counts >= 2].index
kelas_dibuang = counts[counts < 2].index.tolist()

if kelas_dibuang:
    nama_dibuang = [map_kategori_terbalik[k] for k in kelas_dibuang]
    print(f"Catatan: Kelas {nama_dibuang} dikeluarkan karena sampel terlalu sedikit.")

X = X[y.isin(kelas_valid)]
y = y[y.isin(kelas_valid)]

# Pembagian data 80% latih, 20% uji
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Data latih : {X_train.shape[0]} baris")
print(f"Data uji   : {X_test.shape[0]} baris\n")

# Pelatihan model Random Forest
model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
model.fit(X_train, y_train)
print("Model berhasil dilatih.")

## Tahap 6 — Evaluasi Model

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

print("=== TAHAP 6: EVALUASI MODEL ===")

y_pred = model.predict(X_test)

# Nama kelas sesuai yang ada di data uji
kelas_uji    = sorted(y_test.unique())
target_names = [map_kategori_terbalik[k] for k in kelas_uji]

# Akurasi
print("=" * 50)
print(f"Akurasi Model : {accuracy_score(y_test, y_pred)*100:.2f}%")
print("=" * 50)

# Classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, labels=kelas_uji, target_names=target_names))

# Confusion matrix
sns.set_theme(style="white")
fig, ax = plt.subplots(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred, labels=kelas_uji)
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', ax=ax,
            xticklabels=target_names, yticklabels=target_names,
            annot_kws={"size": 12, "weight": "bold"})
ax.set_title('Confusion Matrix — Prediksi Kualitas Udara ISPU', pad=15, fontsize=13)
ax.set_xlabel('Prediksi Model', fontsize=11)
ax.set_ylabel('Data Aktual', fontsize=11)
plt.tight_layout()
nama_eval = 'confusion_matrix_ispu.png'
plt.savefig(nama_eval, dpi=300, bbox_inches='tight')
plt.show()
print(f"Confusion matrix disimpan: '{nama_eval}'")

## Tahap 7 — Validasi (Stratified K-Fold)

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
import numpy as np

print("=== TAHAP 7: STRATIFIED K-FOLD CROSS VALIDATION ===")

cv_strategi = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
skor_folds  = cross_val_score(model, X, y, cv=cv_strategi, scoring='accuracy')

print("=" * 50)
for i, skor in enumerate(skor_folds, start=1):
    print(f"  Fold {i}: {skor*100:.2f}%")
print("-" * 50)
print(f"  Rata-rata : {skor_folds.mean()*100:.2f}%")
print(f"  Std Dev   : {skor_folds.std()*100:.4f}%")
print("=" * 50)

if skor_folds.mean() >= 0.95 and skor_folds.std() < 0.02:
    print("\nKesimpulan: Model stabil dan akurat.")
    print(f"Rata-rata akurasi {skor_folds.mean()*100:.2f}% dengan std {skor_folds.std()*100:.2f}% "
          f"menunjukkan konsistensi performa pada cross-validation.")
else:
    print("\nKesimpulan: Model menunjukkan fluktuasi akurasi antar fold.")

## Tahap 8 — Simulasi Prediksi Data Baru

In [ ]:
print("=== TAHAP 8: SIMULASI PREDIKSI DATA BARU ===")

# Data simulasi: [pm10, pm25, so2, co, o3, no2]
data_baru = [
    [25.0,  30.0, 12.0,  6.0,  15.0,  8.0],  # → BAIK
    [55.0,  70.0, 25.0, 10.0,  55.0, 15.0],  # → SEDANG
    [75.0,  95.0, 30.0, 15.0, 100.0, 20.0],  # → TIDAK SEHAT
    [85.0, 120.0, 35.0, 18.0, 130.0, 25.0],  # → TIDAK SEHAT (lebih parah)
]

kolom_polutan = ['pm10', 'pm25', 'so2', 'co', 'o3', 'no2']
df_simulasi   = pd.DataFrame(data_baru, columns=kolom_polutan)

# Tambahkan kolom one-hot stasiun (semua 0 = stasiun DKI1 sebagai referensi)
for col in ['stasiun_DKI2', 'stasiun_DKI3', 'stasiun_DKI4', 'stasiun_DKI5']:
    df_simulasi[col] = 0

# Sesuaikan urutan kolom dengan data latih
df_simulasi = df_simulasi[X.columns]

# Prediksi
prediksi          = model.predict(df_simulasi)
prediksi_nama     = [map_kategori_terbalik[k] for k in prediksi]

df_hasil          = pd.DataFrame(data_baru, columns=kolom_polutan)
df_hasil['Prediksi Kategori'] = prediksi_nama

print("Data input:")
print(df_hasil.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

print("=== VISUALISASI PROBABILITAS PREDIKSI ===")

# Hitung probabilitas tiap kelas untuk setiap sampel
y_proba         = model.predict_proba(df_simulasi)
nama_kelas      = [map_kategori_terbalik[c] for c in model.classes_]
warna_kelas     = ['#2ecc71', '#f1c40f', '#e67e22']
warna_mapped    = [warna_kelas[c] for c in model.classes_]

fig, axes = plt.subplots(1, len(df_hasil), figsize=(18, 5))

for i, (proba, pred) in enumerate(zip(y_proba, prediksi)):
    axes[i].bar(nama_kelas, proba, color=warna_mapped, edgecolor='white', linewidth=1.2)
    axes[i].set_title(
        f'Sampel {i+1}\n'
        f'PM10={data_baru[i][0]} | PM2.5={data_baru[i][1]}\n'
        f'→ {map_kategori_terbalik[pred]}',
        fontsize=9, fontweight='bold'
    )
    axes[i].set_ylim(0, 1.15)
    axes[i].set_ylabel('Probabilitas' if i == 0 else '')
    axes[i].tick_params(axis='x', rotation=20, labelsize=9)
    for j, p in enumerate(proba):
        if p > 0.03:
            axes[i].text(j, p + 0.02, f'{p:.2f}',
                         ha='center', fontsize=9, fontweight='bold')

plt.suptitle('Probabilitas Prediksi per Kelas — Simulasi Data Baru',
             fontsize=13, fontweight='bold', y=1.05)
plt.tight_layout()
nama_plot = 'prediksi_data_baru.png'
plt.savefig(nama_plot, dpi=150, bbox_inches='tight')
plt.show()
print(f"Grafik disimpan: '{nama_plot}'")